# Test Open library API — POC Data

Objectif :

- tester Open library API sur une première liste de 20 EAN/ISBN ;
- vérifier la couverture de l’API ;
- analyser rapidement la complétude des métadonnées ;
- capturer les erreurs sans bloquer le notebook ;
- exporter les résultats pour analyse future.

## Import

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.append(str(PROJECT_ROOT / "src"))

from collectionlens.api.open_library_api import search_book_by_isbn

## Liste ISBN test

In [2]:
isbns = [
    "9782351423554",
    "9791042019396",
    "9782820354488",
    "9782290042298",
    "9782845808331",
    "9782413042341",
    "9782384967421",
    "9791043301087",
    "9782374123035",
    "9782374126173",
    "9791039143431",
    "9791026854920",
    "9791039140652",
    "9791026856283",
    "9791039147156",
    "9782203001190",
    "9782858150045",
    "9791038206229",
    "9782822244787",
    "9791038209657",
]

## Fonction de test google books

In [3]:
def build_openlibrary_results(isbns: list[str]) -> pd.DataFrame:
    """Lance un test OpenLibrary sur une liste d'ISBN/EAN.
    Args:
        isbns (list[str]): Liste d'ISBN/EAN à tester.
    Returns:
        pd.DataFrame: DataFrame contenant les résultats du test.
    """
    rows = []

    for isbn in isbns:
        result = search_book_by_isbn(isbn)
        rows.append(result)

    return pd.DataFrame(rows)

## Exécution du test

In [4]:

df_openlibrary = build_openlibrary_results(isbns)

df_openlibrary

,source,isbn_query,found,error,status_code,source_id,openlibrary_key,isbn,title,subtitle,...,page_count,print_type,maturity_rating,average_rating,ratings_count,preview_link,info_link,canonical_volume_link,industry_identifiers,raw_data
0,openlibrary,9782351423554,True,None,200,/books/OL57621241M,/books/OL57621241M,9782351423554,Vinland Saga (1),None,...,NaN,BOOK,NaN,NaN,NaN,http://openlibrary.org/books/OL57621241M/Vinla...,http://openlibrary.org/books/OL57621241M/Vinla...,http://openlibrary.org/books/OL57621241M/Vinla...,"{'isbn_10': ['2351423550'], 'isbn_13': ['97823...",{'url': 'http://openlibrary.org/books/OL576212...
1,openlibrary,9791042019396,False,no_result,200,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,openlibrary,9782820354488,False,no_result,200,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,openlibrary,9782290042298,True,None,200,/books/OL12514287M,/books/OL12514287M,9782290042298,"Fly, tome 1",Le Précepteur du héros,...,NaN,BOOK,NaN,NaN,NaN,http://openlibrary.org/books/OL12514287M/Fly_t...,http://openlibrary.org/books/OL12514287M/Fly_t...,http://openlibrary.org/books/OL12514287M/Fly_t...,"{'goodreads': ['5118323'], 'librarything': ['8...",{'url': 'http://openlibrary.org/books/OL125142...
4,openlibrary,9782845808331,False,no_result,200,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,openlibrary,9782413042341,False,no_result,200,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,openlibrary,9782384967421,False,no_result,200,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,openlibrary,9791043301087,False,no_result,200,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,openlibrary,9782374123035,False,no_result,200,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,openlibrary,9782374126173,True,None,200,/books/OL61173846M,/books/OL61173846M,9782374126173,"Sinbad, Tome 1",None,...,226.0,BOOK,NaN,NaN,NaN,http://openlibrary.org/books/OL61173846M/Sinba...,http://openlibrary.org/books/OL61173846M/Sinba...,http://openlibrary.org/books/OL61173846M/Sinba...,"{'isbn_13': ['9782374126173'], 'openlibrary': ...",{'url': 'http://openlibrary.org/books/OL611738...


## Analyse rapide

In [5]:

coverage_rate = df_openlibrary["found"].mean() * 100

print(f"Taux de couverture OpenLibrary : {coverage_rate:.2f}%")

Taux de couverture OpenLibrary : 20.00%


In [6]:

df_openlibrary["found"].value_counts(dropna=False)

found
False    16
True      4
Name: count, dtype: int64

## Analyse des erreurs

In [7]:
df_openlibrary.loc[~df_openlibrary["found"], ["isbn_query", "error"]]

,isbn_query,error
1,9791042019396,no_result
2,9782820354488,no_result
4,9782845808331,no_result
5,9782413042341,no_result
6,9782384967421,no_result
7,9791043301087,no_result
8,9782374123035,no_result
10,9791039143431,no_result
11,9791026854920,no_result
12,9791039140652,no_result


## Complétude des métadonnées

In [9]:

metadata_fields = [
    "title",
    "authors",
    "publisher",
    "published_date",
    "language",
    "description",
    "categories",
    "thumbnail",
]

available_fields = [
    field for field in metadata_fields
    if field in df_openlibrary.columns
]

metadata_completeness = (
    df_openlibrary[available_fields]
    .notna()
    .mean()
    .mul(100)
    .round(2)
    .reset_index()
)

metadata_completeness.columns = ["field", "completion_rate"]

metadata_completeness

,field,completion_rate
0,title,20.0
1,authors,20.0
2,publisher,20.0
3,published_date,20.0
4,language,0.0
5,description,0.0
6,categories,20.0
7,thumbnail,15.0


In [ ]:
df_openlibrary_ok = df_openlibrary[df_openlibrary["found"]].copy()
df_openlibrary_ko = df_openlibrary[~df_openlibrary["found"]].copy()

display(df_openlibrary_ok)
display(df_openlibrary_ko)

In [ ]:

output_dir = PROJECT_ROOT / "data" / "processed" / "openlibrary"
output_dir.mkdir(parents=True, exist_ok=True)

full_output_path = output_dir / "openlibrary_results_full.csv"
ok_output_path = output_dir / "openlibrary_results_ok.csv"
ko_output_path = output_dir / "openlibrary_results_ko.csv"

df_openlibrary.to_csv(full_output_path, index=False)
df_openlibrary_ok.to_csv(ok_output_path, index=False)
df_openlibrary_ko.to_csv(ko_output_path, index=False)


```markdown
# Premiers constats

Les premiers tests réalisés sur OpenLibrary API montrent des limites importantes dans le cadre du POC CollectionLens.

## Couverture des ouvrages

Le taux de couverture observé sur l’échantillon testé est d’environ 20 %, ce qui reste très faible pour une utilisation comme source principale de métadonnées.

Une grande partie des ISBN manga, BD et comics français récents n’a pas été retrouvée.

## Complétude des métadonnées

Les métadonnées récupérées sont globalement moins complètes que celles obtenues via Google Books.

Les champs suivants restent partiellement renseignés :
- titre ;
- auteurs ;
- éditeur ;
- date de publication ;
- catégories.

Plusieurs champs importants sont quasiment absents :
- langue ;
- description ;
- miniature (thumbnail).

## Limites identifiées

Les limites suivantes ont été observées :
- très faible couverture des ISBN ;
- faible présence des éditions françaises récentes ;
- descriptions rarement disponibles ;
- métadonnées souvent incomplètes ;
- couverture limitée pour mangas, BD et comics.

## Conclusion POC

OpenLibrary API peut constituer une source complémentaire intéressante dans certains cas, mais sa couverture et sa complétude apparaissent insuffisantes pour répondre seule aux besoins de CollectionLens.

Les premiers benchmarks confirment la nécessité :
- d’une stratégie multi-sources ;
- d’un pipeline de normalisation ;
- d’un futur système de cache local ;
- de l’évaluation d’autres sources bibliographiques spécialisées.
```



